# Detecting Mouse faces pain/not pain using PyTorch

Code copied from:
Image classification of Chest X Rays in one of three classes: Normal, Viral Pneumonia, COVID-19

Notebook created for the guided project [Detecting COVID-19 with Chest X Ray using PyTorch](https://www.coursera.org/projects/covid-19-detection-x-ray) on Coursera

Dataset from [COVID-19 Radiography Dataset](https://www.kaggle.com/tawsifurrahman/covid19-radiography-database) on Kaggle

# Importing Libraries

In [ ]:
# !pip3 install tqdm

In [ ]:
%matplotlib inline

import os, copy
import shutil
import random
import torch
import torchvision
import numpy as np

import pandas as pd

from PIL import Image
from matplotlib import pyplot as plt

torch.manual_seed(0)

print('Using PyTorch version', torch.__version__)

### Preparing Training and Test Sets

#### HEIC to JPG - https://freetoolonline.com/heic-to-jpg.html

In [ ]:
root_dir = './data'
root_result = './data/results'
type_image = 'jpg';

class_names = ['normal', 'pain']
source_dirs = ['normal', 'pain']

root_normal = os.path.join(root_dir, source_dirs[0])
root_pain   = os.path.join(root_dir, source_dirs[1])

root_backup_normal = os.path.join(root_dir, 'backup', source_dirs[0])
root_backup_pain   = os.path.join(root_dir, 'backup', source_dirs[1])

root_test        = os.path.join(root_dir, 'test')
root_test_normal = os.path.join(root_dir, 'test', source_dirs[0])
root_test_pain   = os.path.join(root_dir, 'test', source_dirs[1])

class_names == source_dirs, type_image, root_pain, root_backup_pain, root_test_pain

In [ ]:
def create_dir(rooti):
    try:
        os.mkdir(rooti)
    except:
        print("it is ok: %s"%(rooti))
        
        
create_dir(root_normal)        
create_dir(root_pain)
create_dir(root_test)
create_dir(root_test_normal)        
create_dir(root_test_pain)

In [ ]:
def clean_data(rooti):
    fnames = [x for x in os.listdir(rooti) ]
    for fname in fnames:
        filefull = os.path.join(rooti, fname)
        os.unlink(filefull)
        
clean_data(root_normal)
clean_data(root_pain)
clean_data(root_test_normal)
clean_data(root_test_pain)

### How many samples

In [ ]:
nnormals = len(os.listdir(root_backup_normal))
npains   = len(os.listdir(root_backup_pain))
print(root_backup_normal, root_backup_pain)

'total images', nnormals, npains

### Sample data

In [ ]:
print(root_backup_normal)
len(os.listdir(root_backup_normal))
term = 'front'
print(term)
ori_images_normal = [x for x in os.listdir(root_backup_normal) if term in x]
ori_images_pain   = [x for x in os.listdir(root_backup_pain) if term in x]
len(ori_images_normal), len(ori_images_pain)

In [ ]:
def copy_backup_to_data(term, negative=False, exclude_list=[], hand=False, blurry=False, type_image = 'jpg'):
    # must be empty to start
    nnormals = len(os.listdir(root_normal))
    npains   = len(os.listdir(root_pain))

    if nnormals + npains != 0:
        print("Already done")
        return
    
    #----------- normal --------------------
    ori_images = [x for x in os.listdir(root_backup_normal) if x.lower().endswith(type_image)]
    ori_images = [x for x in ori_images if 'many' not in x]
    
    if term != 'all':
        if negative:
            ori_images = [x for x in ori_images if term not in x]
        else:
            ori_images = [x for x in ori_images if term in x]
            
    for term2 in exclude_list:
        ori_images = [x for x in ori_images if term2 not in x]
        
        
    if hand is not None:
        if hand == True:
            ori_images = [x for x in ori_images if 'hand' in x]
        else:
            ori_images = [x for x in ori_images if 'hand' not in x]

    if blurry is not None:
        if blurry == True:
            ori_images = [x for x in ori_images if 'blurry' in x]
        else:
            ori_images = [x for x in ori_images if 'blurry' not in x]

    for image in ori_images:
        source_path = os.path.join(root_backup_normal, image)
        target_path = os.path.join(root_normal, image)
        shutil.copy(source_path, target_path) 
        
    #--------- pain -----------------------------
    ori_images = [x for x in os.listdir(root_backup_pain) if x.lower().endswith(type_image)]
    if term != 'all':
        if negative:
            ori_images = [x for x in ori_images if term not in x]
        else:
            ori_images = [x for x in ori_images if term in x]
            
    for term2 in exclude_list:
        ori_images = [x for x in ori_images if term2 not in x]

    if hand is not None:
        if hand == True:
            ori_images = [x for x in ori_images if 'hand' in x]
        else:
            ori_images = [x for x in ori_images if 'hand' not in x]

                        
    if blurry is not None:
        if blurry == True:
            ori_images = [x for x in ori_images if 'blurry' in x]
        else:
            ori_images = [x for x in ori_images if 'blurry' not in x]


    for image in ori_images:
        source_path = os.path.join(root_backup_pain, image)
        target_path = os.path.join(root_pain, image)
        shutil.copy(source_path, target_path)        


In [ ]:
clean_data(root_normal)
clean_data(root_pain)
clean_data(root_test_normal)
clean_data(root_test_pain)

blurry=False; hand=None
negative=False

term='all'; exclude_list = []
term='side'; exclude_list = []
term='side'; exclude_list = ['top']
term='front'; exclude_list = []
term='front'; exclude_list = ['top']
term='top'; exclude_list = []
term='top'; exclude_list = ['front']
term='top'; exclude_list = ['side']

copy_backup_to_data(term=term, negative=negative, exclude_list=exclude_list, hand=hand, blurry=blurry, type_image = 'jpg')  

In [ ]:
len(os.listdir(root_normal)), len(os.listdir(root_pain))

In [ ]:
perc_test=.5

def move_random_samples_to_test(perc=.5):
    
    nnormals = len(os.listdir(root_test_normal))
    npains   = len(os.listdir(root_test_pain))

    if nnormals + npains != 0:
        print("Already done")
        return
    
    #----------- normal --------------------
    ori_images = [x for x in os.listdir(root_normal) ]
    global num_test_normal_imgs
    num_test_normal_imgs = int(len(ori_images)*perc_test)
    sel_images = random.sample(ori_images, num_test_normal_imgs)
    
    for image in sel_images:
        source_path = os.path.join(root_normal, image)
        target_path = os.path.join(root_test_normal, image)
        shutil.move(source_path, target_path)
        
    #--------- pain -----------------------------
    ori_images = [x for x in os.listdir(root_pain) ]
    global num_test_pain_imgs
    num_test_pain_imgs = int(len(ori_images)*perc_test)
    sel_images = random.sample(ori_images, num_test_pain_imgs)
    
    for image in sel_images:
        source_path = os.path.join(root_pain, image)
        target_path = os.path.join(root_test_pain, image)
        shutil.move(source_path, target_path)       

move_random_samples_to_test()   
num_test_normal_imgs, num_test_pain_imgs

# Creating Custom Dataset

### Original size

In [ ]:
fnames = [x for x in os.listdir(root_pain) ]

im = Image.open(os.path.join(root_pain, fnames[0]))
width, height = im.size
ratio = np.round(width/height,2)
print(width, height, ratio)

# width, height = 4032, 2268
print(width, height)

In [ ]:
class ImageDataset(torch.utils.data.Dataset):
    
    def __init__(self, image_dirs, transform):
        def get_images(class_name):
            images = [x for x in os.listdir(image_dirs[class_name]) if x.lower().endswith(type_image)]
            print(f'Found {len(images)} {class_name} examples')
            return images
        
        self.images = {}
        self.class_names = ['normal', 'pain']
        
        for class_name in self.class_names:
            self.images[class_name] = get_images(class_name)
            
        self.image_dirs = image_dirs
        self.transform = transform
        
    
    def __len__(self):
        return sum([len(self.images[class_name]) for class_name in self.class_names])
    
    
    def __getitem__(self, index):
        class_name = random.choice(self.class_names)
        index = index % len(self.images[class_name])
        image_name = self.images[class_name][index]
        image_path = os.path.join(self.image_dirs[class_name], image_name)
        image = Image.open(image_path).convert('RGB')
        return self.transform(image), self.class_names.index(class_name)

# Image Transformations

In [ ]:
# height = 224
height = 1200
height = 600
width = int(height*ratio)

img_size = (height,width) # ???
print(img_size)

train_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=img_size),
    torchvision.transforms.RandomHorizontalFlip(),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = torchvision.transforms.Compose([
    torchvision.transforms.Resize(size=img_size),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Prepare DataLoader

In [ ]:
train_dirs = {
    'normal': root_normal,
    'pain': root_pain
}

train_dataset = ImageDataset(train_dirs, train_transform)

In [ ]:
test_dirs = {
    'normal': root_test_normal,
    'pain': root_test_pain
}

test_dataset = ImageDataset(test_dirs, test_transform)

In [ ]:
batch_size = 6

dl_train = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
dl_test  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

print('Number of training batches', len(dl_train))
print('Number of test batches', len(dl_test))

# Data Visualization

In [ ]:
class_names = train_dataset.class_names

def show_images(images, labels, preds):
    plt.figure(figsize=(32, 24))
    for i, image in enumerate(images):
        plt.subplot(1, 6, i + 1, xticks=[], yticks=[])
        image = image.numpy().transpose((1, 2, 0))
        mean = np.array([0.485, 0.456, 0.406])
        std  = np.array([0.229, 0.224, 0.225])
        image = image * std + mean
        image = np.clip(image, 0., 1.)
        plt.imshow(image)

        # print(i, preds[i], labels[i])
        col = 'red' if (preds[i] != labels[i]) else 'green'
            
        plt.xlabel(f'{class_names[int(labels[i].numpy())]}', fontsize=32)
        plt.ylabel(f'{class_names[int(preds[i].numpy())]}', color=col, fontsize=32)
    plt.tight_layout()
    plt.show()

In [ ]:
images, labels = next(iter(dl_train))
print(len(images), labels, labels)
show_images(images, labels, labels)

In [ ]:
images, labels = next(iter(dl_test))
print(len(images), labels, labels)
show_images(images, labels, labels)

In [ ]:
len(dl_test)

In [ ]:
#for images, labels in dl_test:
#    print(len(images), labels)

In [ ]:
width, height, width * height

# Creating the Model

In [ ]:
resnet18 = torchvision.models.resnet18(pretrained=True)
# print(resnet18)

In [ ]:
resnet18.fc = torch.nn.Linear(in_features=512, out_features=len(class_names))
loss_fn = torch.nn.CrossEntropyLoss()

# bad - optimizer = torch.optim.Adam(resnet18.parameters(), lr=1e-8)
lr=1e-5
optimizer = torch.optim.Adam(resnet18.parameters(), lr=lr)

from torch.optim.lr_scheduler import StepLR
# scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.1)
scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

In [ ]:
def show_preds():
    resnet18.eval()
    images, labels = next(iter(dl_test))
    outputs = resnet18(images)
    _, preds = torch.max(outputs, 1)
    print(len(images), labels, preds)
    show_images(images, labels, preds)

In [ ]:
show_preds()

# Training the Model

In [ ]:
fname_best_model= './data/best_model_weights.pth'
loss_list = []; mean_accuracy_list=[]; lr_list=[]; dic= {}

def train(epochs):
    print('Starting training..')
    global train_loss_list
    train_loss_list = []
    
    global mean_accuracy_list
    mean_accuracy_list=[]
    
    global lr_list
    lr_list=[]
    
    global dic
    dic= {}
    
    count_resnet=-1; max_accuracy = 0; min_loss=9999; count=-1

    for e in range(epochs):
        print('='*20)
        print(f'Starting epoch {e + 1}/{epochs}')
        print('='*20)

        train_loss = 0.
        val_loss = 0.

        resnet18.train() # set model to training phase
            
        accuracy_list = []

        for nTrain, (images, labels) in enumerate(dl_train):
            count += 1
            
            optimizer.zero_grad()
            outputs = resnet18(images)
            loss = loss_fn(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
            if nTrain % 10 == 0:
                print('Evaluating at step', nTrain)

                accuracy = 0
                resnet18.eval() # set model to eval phase
                for val_step, (images, labels) in enumerate(dl_test):
                    outputs = resnet18(images)
                    loss = loss_fn(outputs, labels)
                    val_loss += loss.item()

                    _, preds = torch.max(outputs, 1)
                    accuracy += np.sum((preds == labels).numpy())

                val_loss /= (val_step + 1)
                accuracy = accuracy/len(test_dataset)
                accuracy_list.append(accuracy)
                lr=optimizer.param_groups[0]["lr"]
                
                print(f'Validation Loss: {val_loss:.4f}, Accuracy: {accuracy:.4f}, LR: {lr:.2e}')

                # show_preds()

                resnet18.train()

                if accuracy >= 0.7:
                    loss = train_loss / (nTrain + 1)

                    #if ((accuracy >= max_accuracy) or (loss < min_loss)) or (accuracy >= 0.95):
                        
                    #    if (accuracy >= max_accuracy):
                    ''' PyTorch models store the learned parameters in an internal state dictionary, 
                        called state_dict. These can be persisted via the torch.save method '''
                    # torch.save(resnet18.state_dict(), fname_best_model)
                    if accuracy >= max_accuracy: max_accuracy = accuracy
                    if loss < min_loss:          min_loss     = loss
                    resnetx = resnet18.state_dict()
                    count_resnet += 1
                    dic[count_resnet]  = {}
                    dic[count_resnet]['count'] = count
                    dic[count_resnet]['epoch'] = e
                    dic[count_resnet]['lr'] = np.log10(lr)
                    dic[count_resnet]['accuracy'] = accuracy
                    dic[count_resnet]['loss'] = loss
                    dic[count_resnet]['resnet'] = resnetx
                    print(">>> %d) max accuracy %.2f, min loss %.5f "%(count_resnet, max_accuracy*100, min_loss) )

        mean_accuracy = np.mean(accuracy_list)
        train_loss /= (nTrain + 1)
        
        lr_list.append(lr)

        print(f'Training Loss: {train_loss:.4f}')
        train_loss_list.append(train_loss)
        mean_accuracy_list.append(mean_accuracy)
        
        scheduler.step()
        
    print('Training complete..')
    return # train_loss_list, mean_accuracy_list, lr_list, dic

In [ ]:
term, exclude_list

In [ ]:
%%time
# train_loss_list, mean_accuracy_list, lr_list, dic = train(epochs=30)
epochs = 50; times=6
train(epochs=epochs)

# Final Results

In [ ]:
df = pd.DataFrame(dic).T
print(len(df))
df = df.sort_values("accuracy", ascending=False)
df.index = np.arange(0, len(df))

In [ ]:
df[['lr', 'accuracy', 'loss',]].head(5)

In [ ]:
lr

In [ ]:
fname0 = 'resnet_%s_%d_epochs_%d_lr_%.2e_batch_size_%d_term_%s_num_test_img_%d_negative_%s_exclude_%s_hand_%s_blurry_%s.tsv'
fname = fname0%('RESULT', times, epochs, lr, batch_size, term, num_test_pain_imgs, negative, str(exclude_list), hand, blurry)
df.to_csv(os.path.join(root_result, fname), sep='\t')
fname, root_dir

In [ ]:
df2 = pd.DataFrame({"epochs": np.arange(0, len(train_loss_list))+1, 'loss': train_loss_list})
fname = fname0%('LOSS', times, epochs, lr, batch_size, term, num_test_pain_imgs, negative, str(exclude_list), hand, blurry)
df2.to_csv(os.path.join(root_result, fname), sep='\t')

In [ ]:
df3 = pd.DataFrame({"epochs": np.arange(0, len(mean_accuracy_list))+1, 'loss': mean_accuracy_list})
fname = fname0%('ACCURACY', times, epochs, lr, batch_size, term, num_test_pain_imgs, negative, str(exclude_list), hand, blurry)
df3.to_csv(os.path.join(root_result, fname), sep='\t')

In [ ]:
# current, last model
show_preds()

In [ ]:
# save current, last model
resnet18_backup = copy.deepcopy(resnet18)

# load best model
#resnet18.load_state_dict(torch.load(fname_best_model))
#resnet18.eval()

In [ ]:
df.columns

In [ ]:
lr, accu, loss = df.iloc[0][['lr', 'accuracy', 'loss']]
lr, accu, loss

In [ ]:
for i in range(5):
    resnet18.load_state_dict(df.iloc[i].resnet)
    total, totok = 0, 0
    
    for images, labels in dl_test:
        outputs = resnet18(images)
        _, preds = torch.max(outputs, 1)
        ok = np.sum([labels[i] == preds[i] for i in range(len(labels))])
        total += len(labels)
        totok += ok    
        # print(totok, total, len(images), labels, preds)
        show_images(images, labels, preds)

    lr, accu, loss = df.iloc[i][['lr', 'accuracy', 'loss']]
    print("%d) lr 1e%.3f, pred.accuracy: %.2f%%, loss: %.3f"%(i, lr, 100.*accu, loss))
    print("%d) percentage test accuracy: %.2f%%, ok: %d, total: %d"%(i, 100.*totok/total, totok, total))
    print("------------------------------------------")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline


plt.plot(train_loss_list, linestyle = 'dotted')
plt.xlabel("epochs", fontsize=14)
plt.ylabel("loss", fontsize=14)


fname_png = fname0.replace(".tsv", ".png")
fname = fname_png%('LOSS', times, epochs, lr, batch_size, term, num_test_pain_imgs, negative, str(exclude_list), hand, blurry)
fnameful = os.path.join(root_result, fname)
print(fnameful)
plt.savefig(fnameful, dpi=300, facecolor='white')

plt.show()

In [ ]:
plt.plot([np.log10(x) for x in lr_list], linestyle = 'dotted')
plt.xlabel("epochs", fontsize=14)
plt.ylabel("lr", fontsize=14)

fname_png = fname0.replace(".tsv", ".png")
fname = fname_png%('lr', times, epochs, lr, batch_size, term, num_test_pain_imgs, negative, str(exclude_list), hand, blurry)
fnameful = os.path.join(root_result, fname)
plt.savefig(fnameful, dpi=300, facecolor='white')

plt.show()


In [ ]:
plt.plot(mean_accuracy_list, linestyle = 'dotted')
plt.xlabel("epochs", fontsize=14)
plt.ylabel("accuracy in %", fontsize=14)

fname_png = fname0.replace(".tsv", ".png")
fname = fname_png%('ACCURACY', times, epochs, lr, batch_size, term, num_test_pain_imgs, negative, str(exclude_list), hand, blurry)
fnameful = os.path.join(root_result, fname)
plt.savefig(fnameful, dpi=300, facecolor='white')

plt.show()